# Solutions for Week 9 activities: APIs and Web Scraping 

# Setup

In [1]:
## imports
import pandas as pd
import numpy as np
import re
import requests
from bs4 import BeautifulSoup  
import yaml
import os

## repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## census wrapper and helper package with state FIPS codes
## if not installed: python -m pip install census us
from census import Census
from us import states

## print all of text
pd.options.display.max_colwidth = 1000

## Activity 1: NAEP API (no credentials needed)

Site: National Assessment of Education Progress (NAEP)

Documentation: https://www.nationsreportcard.gov/api_documentation.aspx

Base link: https://www.nationsreportcard.gov/DataService/GetAdhocData.aspx 

A. Using their example query of 2011 writing scores separated by gender
based on here - https://stackoverflow.com/questions/40836749/pythonic-way-of-writing-a-single-line-long-string

Use the ( ) syntax to formulate a long string stored as `example_naep_query` without linebreaks added

In [2]:
## using their example query of 2011 writing scores separated by gender
## based on here - https://stackoverflow.com/questions/40836749/pythonic-way-of-writing-a-single-line-long-string
## using the ( ) syntax to formulate a long
## string without linebreaks added
example_naep_query = (
'https://www.nationsreportcard.gov/'
'Dataservice/GetAdhocData.aspx?'
'type=data&subject=writing&grade=8&'
'subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011')


example_naep_query


'https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?type=data&subject=writing&grade=8&subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011'

B. Use requests to call the API and print the response

In [3]:
naep_resp = requests.get(example_naep_query)
naep_resp
print(type(naep_resp))



<Response [200]>

<class 'requests.models.Response'>


C. Get the json contents of the response

In [4]:
naep_resp_j = naep_resp.json()
naep_resp_j


{'status': 200,
 'serviceVersion': '6.4.2026.1',
 'dwellTimeMS': '15.6803',
 'avgWebHostCPUTotalLoad': 'N/A',
 'dataHitType': 'FROM_MEMORY',
 'Source': 'B11A',
 'result': [{'year': 2011,
   'sample': 'R3',
   'yearSampleLabel': '2011',
   'Cohort': 2,
   'CohortLabel': 'Grade 8',
   'stattype': 'MN:MN',
   'subject': 'WRI',
   'grade': 8,
   'scale': 'WRIRP',
   'jurisdiction': 'NP',
   'jurisLabel': 'National public',
   'variable': 'GENDER',
   'variableLabel': 'Sex',
   'varValue': '1',
   'varValueLabel': 'Male',
   'value': 139.099504632971,
   'isStatDisplayable': 1,
   'errorFlag': 0},
  {'year': 2011,
   'sample': 'R3',
   'yearSampleLabel': '2011',
   'Cohort': 2,
   'CohortLabel': 'Grade 8',
   'stattype': 'MN:MN',
   'subject': 'WRI',
   'grade': 8,
   'scale': 'WRIRP',
   'jurisdiction': 'NP',
   'jurisLabel': 'National public',
   'variable': 'GENDER',
   'variableLabel': 'Sex',
   'varValue': '2',
   'varValueLabel': 'Female',
   'value': 158.567104984955,
   'isStatDispl

D. Extract the result from that json and turn it into a dataframe

In [5]:

## with result, turn it into a dataframe
naep_resp_d = pd.DataFrame(naep_resp_j['result'])
naep_resp_d

,year,sample,yearSampleLabel,Cohort,CohortLabel,stattype,subject,grade,scale,jurisdiction,jurisLabel,variable,variableLabel,varValue,varValueLabel,value,isStatDisplayable,errorFlag
0,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,1,Male,139.099505,1,0
1,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,2,Female,158.567105,1,0


E. Run code to see what happens with an error in the query

In [6]:
## here's a query that from the documentation we know
## won't work since i modified year to 2025 which doesnt
## exist in the data
wrong_naep_query = (
'https://www.nationsreportcard.gov/'
'Dataservice/GetAdhocData.aspx?'
'type=data&subject=writing&grade=8&'
'subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2025')

wrong_naep_query

'https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?type=data&subject=writing&grade=8&subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2025'

In [7]:
## use requests to call the api
naep_wrong_resp = requests.get(wrong_naep_query)
naep_wrong_resp

## in the case of this particular api,
## the call returns some response but
## when we try to extract the json containing
## status or results, we get in an error
## uncomment to see error
# naep_wrong_resp.json()

<Response [400]>

In [8]:
## putting it in a try; except as general error catching
try:
    results = naep_wrong_resp.json()['result']
except:
    pass

In [9]:
## if we wanted do more specific error catching,
## see that the status == 400 actually appears here
## so could write if else along thos elines
naep_wrong_resp.text
naep_resp.text

if "System.Exception" in naep_wrong_resp.text:
    print("NAEP results not found")

'{"statusCode":400,"result": "System.Exception: The query \'SELECT DISTINCT Framework FROM Cycles WHERE Subject=\'WRI\' AND Cohort=2 AND CONVERT(VARCHAR(10),Year)+Sample IN (\'2025R3\')\' did not return exactly 1 framework. Make sure you can trend the years defined for the given subject and cohort.\r\n   at NRCDataService3.GetAdhocData.GetFramework(NDEContext& ndeContext, String subjectCode, List`1 yearSamples, String cohort) in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 2733\r\n   at NRCDataService3.GetAdhocData.PopulateBaseOrchestratorRequest() in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 2348\r\n   at NRCDataService3.GetAdhocData.ConstructRequest_Datapoint() in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 947\r\n   at NRCDataService3.GetAdhocData.Page_Load(Object sender, EventArgs e) in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 354"}'

'{"status":200,"serviceVersion":"6.4.2026.1","dwellTimeMS":"15.6803","avgWebHostCPUTotalLoad":"N/A","dataHitType":"FROM_MEMORY","Source":"B11A","result": [{"year":2011,"sample":"R3","yearSampleLabel":"2011","Cohort":2,"CohortLabel":"Grade 8","stattype":"MN:MN","subject":"WRI","grade":8,"scale":"WRIRP","jurisdiction":"NP","jurisLabel":"National public","variable":"GENDER","variableLabel":"Sex","varValue":"1","varValueLabel":"Male","value":139.099504632971,"isStatDisplayable":1,"errorFlag":0},{"year":2011,"sample":"R3","yearSampleLabel":"2011","Cohort":2,"CohortLabel":"Grade 8","stattype":"MN:MN","subject":"WRI","grade":8,"scale":"WRIRP","jurisdiction":"NP","jurisLabel":"National public","variable":"GENDER","variableLabel":"Sex","varValue":"2","varValueLabel":"Female","value":158.567104984955,"isStatDisplayable":1,"errorFlag":0}]}'

NAEP results not found


F. Write a function to make repeated calls to the API

- Say we want to pull the data for grades 4, 8, and 12

Task: write a function called `pull_one_grade` that iterates over a list of those grades and pulls the data for each grade?

**Note**: an ideal function would have arguments for each parameter in the API like subject, subscale, etc. Here we can leave those other parts constant

In [10]:
def pull_one_grade(one_grade):
    
    ## first construct the query 
    q = (
    'https://www.nationsreportcard.gov/'
    'Dataservice/GetAdhocData.aspx?'
    'type=data&subject=writing&grade='+str(one_grade)+'&'+
    'subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011')
    
    ## then, use requests to call
    resp = requests.get(q)
    
    ## next, transform into json
    try:
        resp_j = resp.json()
        resp_df = pd.DataFrame(resp_j['result'])
        return(resp_df)
    except:
        pass  


all_grades = [pull_one_grade(grade) for grade in [4, 8, 12]]
all_grades_df = pd.concat(all_grades)
all_grades_df


,year,sample,yearSampleLabel,Cohort,CohortLabel,stattype,subject,grade,scale,jurisdiction,jurisLabel,variable,variableLabel,varValue,varValueLabel,value,isStatDisplayable,errorFlag
0,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,1,Male,139.099505,1,0
1,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,2,Female,158.567105,1,0
0,2011,R3,2011,3,Grade 12,MN:MN,WRI,12,WRIRP,NP,National public,GENDER,Sex,1,Male,141.256978,1,0
1,2011,R3,2011,3,Grade 12,MN:MN,WRI,12,WRIRP,NP,National public,GENDER,Sex,2,Female,155.385917,1,0


## Activity 2: API for Yelp

In [12]:
## load creds- STUDENTS: REPLACE WITH PATHNAME TO YOURS 
with open("../creds.yaml", 'r') as stream:
    try:
        creds = yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        print(exc)

## get the key
API_KEY = creds['yelp_api']['api_key']
API_KEY

'tupLaqG5g1TDLaoo5wEJqy01MP4ifMqX_KorZq3cCIFYma1DPiqOf2wE-z1JsNBBcvDzrZbPdDU1Okd5DG9N1CwZvU2a79998PjmQdje1KKk00rLOCBW0bCJe7zXanYx'

In [13]:
## function you can use to clean the results
def clean_yelp_json(one_biz):
    """
    Function that cleans the json file with a successful call to the Yelp API
    Inputs:
    one_biz: json entry for one business
    Returns:
    df_str: a dataframe with the values the API returns that are strings
    """
    d_str = {key:value for key, value in one_biz.items()
             if type(value) == str}
    df_str = pd.DataFrame(d_str, index = [d_str['id']])
    return(df_str)


A. Try running a business search query for another place in the United States by constructing a query similar to `yelp_genquery` but changing the location parameter


In [14]:
## query for another location
base_url = "https://api.yelp.com/v3/businesses/search?"
my_name = "restaurants"
my_location = "Burlington,VT,08540"
yelp_genquery = ('{base_url}'
                'term={name}'
                '&location={loc}').format(base_url = base_url,
                name = my_name,
                loc = my_location)

header = {'Authorization': f"Bearer {API_KEY}"}
yelp_genresp = requests.get(yelp_genquery, headers = header).json()



In [15]:
yelp_genresp_list = [clean_yelp_json(one_b) for one_b in yelp_genresp['businesses']]
yelp_stronly_df = pd.concat(yelp_genresp_list)
yelp_stronly_df.head()

,id,alias,name,image_url,url,price,phone,display_phone
IeRVd34QMvQAVNtPpY9ImQ,IeRVd34QMvQAVNtPpY9ImQ,honey-road-burlington,Honey Road,https://s3-media0.fl.yelpcdn.com/bphoto/MAdMFVDb24FaEG2vXfadtA/o.jpg,https://www.yelp.com/biz/honey-road-burlington?adjust_creative=fZtFlUvyaR-hVIafscUD-Q&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=fZtFlUvyaR-hVIafscUD-Q,$$,+18024972145,(802) 497-2145
koImW32HcVxshIQAewdaZQ,koImW32HcVxshIQAewdaZQ,may-day-burlington,May Day,https://s3-media0.fl.yelpcdn.com/bphoto/VaOCaJYD-cvjxekDgqJLQw/o.jpg,https://www.yelp.com/biz/may-day-burlington?adjust_creative=fZtFlUvyaR-hVIafscUD-Q&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=fZtFlUvyaR-hVIafscUD-Q,NaN,+18025409240,(802) 540-9240
-ASe09az4o1j4z9Y628ejg,-ASe09az4o1j4z9Y628ejg,the-farmhouse-tap-and-grill-burlington-2,The Farmhouse Tap & Grill,https://s3-media0.fl.yelpcdn.com/bphoto/pvWKj9bbWRJ5N2xvV7loqQ/o.jpg,https://www.yelp.com/biz/the-farmhouse-tap-and-grill-burlington-2?adjust_creative=fZtFlUvyaR-hVIafscUD-Q&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=fZtFlUvyaR-hVIafscUD-Q,$$,+18028590888,(802) 859-0888
kuyCA_1tbf6kSijRB6qcXA,kuyCA_1tbf6kSijRB6qcXA,hen-of-the-wood-burlington-burlington-4,Hen of the Wood - Burlington,https://s3-media0.fl.yelpcdn.com/bphoto/FYXPofnf7W9oSKuhn2gXmw/o.jpg,https://www.yelp.com/biz/hen-of-the-wood-burlington-burlington-4?adjust_creative=fZtFlUvyaR-hVIafscUD-Q&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=fZtFlUvyaR-hVIafscUD-Q,$$$,+18025400534,(802) 540-0534
S5LfttrH-Pr0Hucz-7EzWQ,S5LfttrH-Pr0Hucz-7EzWQ,frankies-burlington,Frankie's,https://s3-media0.fl.yelpcdn.com/bphoto/LgjqbiFrCHOkwfqAhWaVAg/o.jpg,https://www.yelp.com/biz/frankies-burlington?adjust_creative=fZtFlUvyaR-hVIafscUD-Q&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=fZtFlUvyaR-hVIafscUD-Q,NaN,+18022647094,(802) 264-7094


B. Use the documentation for the business details endpoint to pull details about a specific business

Documentation: https://docs.developer.yelp.com/reference/v3_business_info

In [18]:

hen_id = yelp_stronly_df.id[yelp_stronly_df.name == "Hen of the Wood - Burlington"].iloc[0]
hen_id


'kuyCA_1tbf6kSijRB6qcXA'

In [19]:
## query to get reviews for a specific id
b_base = "https://api.yelp.com/v3/businesses/"
b_query = ('{base_url}'
                '{yelp_id}').format(base_url = b_base,
                yelp_id = hen_id)
b_query

b_json = requests.get(b_query, headers = header).json()
b_json



'https://api.yelp.com/v3/businesses/kuyCA_1tbf6kSijRB6qcXA'

{'id': 'kuyCA_1tbf6kSijRB6qcXA',
 'alias': 'hen-of-the-wood-burlington-burlington-4',
 'name': 'Hen of the Wood - Burlington',
 'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/FYXPofnf7W9oSKuhn2gXmw/o.jpg',
 'is_claimed': True,
 'is_closed': False,
 'url': 'https://www.yelp.com/biz/hen-of-the-wood-burlington-burlington-4?adjust_creative=fZtFlUvyaR-hVIafscUD-Q&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_lookup&utm_source=fZtFlUvyaR-hVIafscUD-Q',
 'phone': '+18025400534',
 'display_phone': '(802) 540-0534',
 'review_count': 953,
 'categories': [{'alias': 'newamerican', 'title': 'New American'}],
 'rating': 4.4,
 'location': {'address1': '55 Cherry St',
  'address2': '',
  'address3': '',
  'city': 'Burlington',
  'zip_code': '05401',
  'country': 'US',
  'state': 'VT',
  'display_address': ['55 Cherry St', 'Burlington, VT 05401'],
  'cross_streets': ''},
 'coordinates': {'latitude': 44.4790764016959, 'longitude': -73.2176419641316},
 'photos': [],
 'price': '$$$',
 'hours':

## Activity 3: Census API (credentials and a wrapper)

API documentation: https://www.census.gov/data/developers.html

Wrapper documentation: https://pypi.org/project/census/

Variable list (ACS 5-year, 2024): https://api.census.gov/data/2024/acs/acs5/variables.html

A. Load the yaml file with your API key for the Census and use the `Census` command to establish a connection

In [16]:
## load creds- STUDENTS: REPLACE WITH PATHNAME TO YOURS
with open("../creds.yaml", 'r') as stream:
    try:
        creds = yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        print(exc)

## get the key
CENSUS_KEY = creds['census_api']['api_key']

## use the key to create a connection to the API
c = Census(CENSUS_KEY)

B. Pick a state and use the `census` wrapper to pull total population, median household income, and the two poverty variables for all counties in that state

| Variable | Description |
|:---|:---|
| `NAME` | Name of the geography |
| `B01003_001E` | Total population |
| `B19013_001E` | Median household income |
| `B17001_001E` | Population for whom poverty status is determined |
| `B17001_002E` | Population with income below the poverty level |

In [17]:
## store variable codes (keys) and readable names (values) in a dictionary
county_vars = {'NAME': 'name',
               'B01003_001E': 'total_pop',
               'B19013_001E': 'median_hh_income',
               'B17001_001E': 'pov_universe',
               'B17001_002E': 'pov_below'}

## pick a state - here, IL
my_state = states.IL
my_state.fips

'17'

In [18]:
## pull the variables for all counties in that state
## using the ACS 5-year estimates for 2024
county_2024 = c.acs5.state_county(fields = tuple(county_vars.keys()),
                                  state_fips = my_state.fips,
                                  county_fips = Census.ALL,
                                  year = 2024)

## the wrapper returns a list of dictionaries (one per county)
county_2024[0]

## convert to a dataframe and rename columns
county_2024_df = (pd.DataFrame(county_2024)
                  .rename(columns = county_vars))

county_2024_df.head()
county_2024_df.shape

## check for census "sentinel" values: large negative numbers
## the census uses to flag estimates that can't be computed
(county_2024_df[['total_pop', 'median_hh_income', 'pov_universe', 'pov_below']] < 0).sum()

{'NAME': 'Adams County, Illinois',
 'B01003_001E': 64754.0,
 'B19013_001E': 66220.0,
 'B17001_001E': 62964.0,
 'B17001_002E': 8084.0,
 'state': '17',
 'county': '001'}

,name,total_pop,median_hh_income,pov_universe,pov_below,state,county
0,"Adams County, Illinois",64754.0,66220.0,62964.0,8084.0,17,001
1,"Alexander County, Illinois",4875.0,47043.0,4773.0,908.0,17,003
2,"Bond County, Illinois",16716.0,65959.0,14625.0,1591.0,17,005
3,"Boone County, Illinois",53230.0,84571.0,52591.0,5224.0,17,007
4,"Brown County, Illinois",6322.0,67917.0,4380.0,405.0,17,009


(102, 7)

total_pop           0
median_hh_income    0
pov_universe        0
pov_below           0
dtype: int64

C. Construct the poverty rate and find the three counties in the state with the highest and lowest rates

In [19]:
## poverty rate: share of people for whom poverty status
## is determined who have income below the poverty line
county_2024_df['pov_rate'] = county_2024_df.pov_below / county_2024_df.pov_universe

county_2024_df[['name', 'pov_universe', 'pov_below', 'pov_rate']].head()

## three counties with the highest poverty rates
county_2024_df.nlargest(3, 'pov_rate')[['name', 'total_pop',
                                        'median_hh_income', 'pov_rate']]

## three counties with the lowest poverty rates
county_2024_df.nsmallest(3, 'pov_rate')[['name', 'total_pop',
                                         'median_hh_income', 'pov_rate']]

,name,pov_universe,pov_below,pov_rate
0,"Adams County, Illinois",62964.0,8084.0,0.128391
1,"Alexander County, Illinois",4773.0,908.0,0.190237
2,"Bond County, Illinois",14625.0,1591.0,0.108786
3,"Boone County, Illinois",52591.0,5224.0,0.099333
4,"Brown County, Illinois",4380.0,405.0,0.092466


,name,total_pop,median_hh_income,pov_rate
76,"Pulaski County, Illinois",5015.0,42463.0,0.249792
38,"Jackson County, Illinois",53064.0,48763.0,0.209869
84,"Schuyler County, Illinois",6787.0,65948.0,0.207700


,name,total_pop,median_hh_income,pov_rate
66,"Monroe County, Illinois",35036.0,102880.0,0.050211
46,"Kendall County, Illinois",137675.0,111601.0,0.054150
31,"Grundy County, Illinois",53219.0,92235.0,0.059870


D. Repeat the pull for 2019 using the `year` argument, merge the two years on the county FIPS code, and calculate the change in median household income

**Note**: for simplicity, we're just calculating the change in nominal income since each year's median income estimates are in that year's dollars; if comparing things like purchasing power, we'd want to adjust for inflation using an index like the CPI

In [20]:
## same pull, changing the year argument to 2019
## (the 2015-2019 ACS 5-year estimates)
county_2019 = c.acs5.state_county(fields = tuple(county_vars.keys()),
                                  state_fips = my_state.fips,
                                  county_fips = Census.ALL,
                                  year = 2019)

county_2019_df = (pd.DataFrame(county_2019)
                  .rename(columns = county_vars))

county_2019_df.head()

,name,total_pop,median_hh_income,pov_universe,pov_below,state,county
0,"Fayette County, Illinois",21565.0,46650.0,19855.0,3421.0,17,051
1,"Logan County, Illinois",29003.0,57308.0,24979.0,2323.0,17,107
2,"Saline County, Illinois",23994.0,44090.0,23435.0,4936.0,17,165
3,"Lake County, Illinois",701473.0,89427.0,684107.0,54273.0,17,097
4,"Massac County, Illinois",14219.0,47481.0,13918.0,2331.0,17,127


In [21]:
## the state and county FIPS codes come back as strings
## (keeping them as strings preserves leading zeros like "001")
county_2024_df[['state', 'county']].dtypes

## county FIPS codes repeat across states (every state has a county "001"),
## so we merge on both state and county codes
county_both = pd.merge(county_2024_df,
                       county_2019_df[['state', 'county', 'median_hh_income']],
                       how = "outer",
                       on = ['state', 'county'],
                       suffixes = ["_2024", "_2019"],
                       indicator = "merge_status")

## merge diagnostics: did every county appear in both years?
county_both.merge_status.value_counts()
county_both.shape

state     object
county    object
dtype: object

both          102
left_only       0
right_only      0
Name: merge_status, dtype: int64

(102, 10)

In [22]:
## change in median household income (dollars and percent)
county_both['income_change'] = county_both.median_hh_income_2024 - county_both.median_hh_income_2019
county_both['income_pct_change'] = county_both.income_change / county_both.median_hh_income_2019

(county_both[['name', 'median_hh_income_2019', 'median_hh_income_2024',
              'income_change', 'income_pct_change']]
 .sort_values(by = 'income_pct_change', ascending = False))

,name,median_hh_income_2019,median_hh_income_2024,income_change,income_pct_change
75,"Pope County, Illinois",38056.0,60050.0,21994.0,0.577938
6,"Calhoun County, Illinois",63009.0,93203.0,30194.0,0.479201
16,"Crawford County, Illinois",49779.0,71674.0,21895.0,0.439844
24,"Effingham County, Illinois",56685.0,80404.0,23719.0,0.418435
39,"Jasper County, Illinois",54256.0,74755.0,20499.0,0.377820
...,...,...,...,...,...
95,"Wayne County, Illinois",50251.0,55521.0,5270.0,0.104874
4,"Brown County, Illinois",61655.0,67917.0,6262.0,0.101565
90,"Union County, Illinois",50625.0,55728.0,5103.0,0.100800
64,"Menard County, Illinois",74684.0,82176.0,7492.0,0.100316


## Activity 4: Scraping a static website with `requests` and `BeautifulSoup`

In this activity, we'll scrape news stories from Georgetown's news site: https://www.georgetown.edu/news/all-news/

We'll follow the scraping routine from lecture:

- **Step 1:** Find a website with information you want to collect
- **Step 2:** Understand the website (use Inspect or SelectorGadget)
- **Step 3:** Write code to collect one realization of the data (**Part A**)
- **Step 4:** Build a scraper: generalize your code into a function (**Parts B-D**)

Before starting, check the site's `robots.txt` (https://www.georgetown.edu/robots.txt) --- it only disallows WordPress admin pages, so the news pages are fair game. We'll still be kind to the server by pausing between requests.

A. Scrape one article

Use this article: https://www.georgetown.edu/news/chinese-language-joy-of-learning/

1. Use `requests` to get the page and check the status code
2. Create a `BeautifulSoup` object
3. Use Inspect or SelectorGadget to find CSS selectors for, and then extract:
    - the title of the article
    - the date it was published
    - the author
    - the text of the article (all paragraphs joined into one string)
4. Store the url and the four pieces of information in a one-row pandas DataFrame

**Hints:**

- If you `.select("p")`, you'll also grab paragraphs that aren't part of the story (e.g., the address in the page footer). Find a more specific selector for the story's body text
- When you extract the title, date, and author, you'll see some extra text like `"Title: "` that isn't visible on the page. Look at the html to see where it comes from, then remove it

In [23]:
## get access to the website
url = "https://www.georgetown.edu/news/chinese-language-joy-of-learning/"
page = requests.get(url)

## check if you got a connection
page.status_code

## create a bs object
soup = BeautifulSoup(page.content, 'html.parser')

## first attempt at the text: all paragraphs
all_p = soup.select("p")
len(all_p)

## the last one is the page footer, not the story
all_p[-1].get_text()

## more specific css selector: paragraphs inside the article body
body_p = soup.select(".c--article-body-text p")
len(body_p)

## clean and join together with string methods
story_text = "\n".join([p.get_text().strip() for p in body_p])
print(story_text[0:500])

## title: the h1 inside the page title field
title_tag = soup.select(".f--page-title h1")[0]
title_tag

## notice the string reader tag in the title - strip
story_title = title_tag.get_text().replace("Title:", "").strip()

## date published
story_date = soup.select(".date")[0].get_text().strip()
story_date

## author
story_author = soup.select(".author")[0].get_text().strip()
story_author

200

20

'Georgetown University\n37th and O Streets, N.W.\nWashington, D.C. 20057\nP. 202-687-0100'

18

Isabela McNeilly-Anta (SFS’27) is a Georgetown Storyteller and senior in the School of Foreign Service studying international political economy with a certificate in international business diplomacy and a minor in French. She is from New Jersey and loves aviation and long-distance running.
“You’re crazy,” was the general consensus from my mom, roommates and friends. A week earlier, I had all my ducks in a row. The GUExperience gods had smiled upon me: my senior-year schedule covered the remainin


<h1><span class="visually-hidden">Title: </span>I’m About to Graduate and Decided to Learn Chinese. Great Timing, Obviously</h1>

'Date Published:\xa0October 7, 2026'

'Author:\xa0Isabela McNeilly-Anta (SFS’27)'

In [24]:
## store in a one-row dataframe
one_article = pd.DataFrame({"url": url,
                            "title": story_title,
                            "date": story_date,
                            "author": story_author,
                            "text": story_text}, index = [0])
one_article

,url,title,date,author,text
0,https://www.georgetown.edu/news/chinese-language-joy-of-learning/,"I’m About to Graduate and Decided to Learn Chinese. Great Timing, Obviously","Date Published: October 7, 2026",Author: Isabela McNeilly-Anta (SFS’27),"Isabela McNeilly-Anta (SFS’27) is a Georgetown Storyteller and senior in the School of Foreign Service studying international political economy with a certificate in international business diplomacy and a minor in French. She is from New Jersey and loves aviation and long-distance running.\n“You’re crazy,” was the general consensus from my mom, roommates and friends. A week earlier, I had all my ducks in a row. The GUExperience gods had smiled upon me: my senior-year schedule covered the remaining requirements for my SFS core, International Political Economy major, French minor and International Business Diplomacy certificate. In fact, if I really wanted to, I could even graduate a semester early. Life was good.\nThat is until I upended my perfect schedule to enroll in Intensive Beginner Mandarin: six credits, four days a week and a recommended daily 2.5 hours of studying.\nCuriously, the idea to study Mandarin came while helping my younger brother move into college. Despite giving..."


B. Write a function to scrape one article

Write a function called `scrape_gu_article` that:

- Takes one input: the url of a Georgetown news article
- Returns a dictionary with the url, title, date, author, and text of the article

Then test your function on these two articles:

- https://www.georgetown.edu/news/chinese-language-joy-of-learning/
- https://www.georgetown.edu/news/board-directors-fall-2026/

**Hint:** not every article lists an author. What happens to your code from Part A when there's no `.author` element on the page? Write your function so it returns `None` for the author in that case rather than throwing an error.

In [25]:
def scrape_gu_article(url):
    """
    Scrapes one article from the Georgetown news site

    Input:
        url (str): url of a Georgetown news article
    Returns:
        dict with the url, title, date, author, and text of the article
    """
    ## get access to the website
    page = requests.get(url)

    ## check if you got a connection
    if page.status_code != 200:
        print(f"Status {page.status_code} for {url}")
        return None

    ## create a bs object
    soup = BeautifulSoup(page.content, 'html.parser')

    ## remove text that's hidden on the page (e.g., "Title: ")
    for hidden in soup.select(".visually-hidden"):
        hidden.decompose()

    ## title
    story_title = soup.select(".f--page-title h1")[0].get_text().strip()

    ## date
    story_date = soup.select(".date")[0].get_text().strip()

    ## author: not every article has one, so check before indexing
    author_tags = soup.select(".author")
    if len(author_tags) > 0:
        story_author = author_tags[0].get_text().strip()
    else:
        story_author = None

    ## text
    body_p = soup.select(".c--article-body-text p")
    story_text = "\n".join([p.get_text().strip() for p in body_p])

    return {"url": url,
            "title": story_title,
            "date": story_date,
            "author": story_author,
            "text": story_text}

In [26]:
## test on an article with an author
test_1 = scrape_gu_article("https://www.georgetown.edu/news/chinese-language-joy-of-learning/")
{key: value[0:100] if isinstance(value, str) else value for key, value in test_1.items()}

## test on an article without an author
test_2 = scrape_gu_article("https://www.georgetown.edu/news/board-directors-fall-2026/")
{key: value[0:100] if isinstance(value, str) else value for key, value in test_2.items()}

{'url': 'https://www.georgetown.edu/news/chinese-language-joy-of-learning/',
 'title': 'I’m About to Graduate and Decided to Learn Chinese. Great Timing, Obviously',
 'date': 'October 7, 2026',
 'author': 'Isabela McNeilly-Anta (SFS’27)',
 'text': 'Isabela McNeilly-Anta (SFS’27) is a Georgetown Storyteller and senior in the School of Foreign Servi'}

{'url': 'https://www.georgetown.edu/news/board-directors-fall-2026/',
 'title': 'Board of Directors Reviews New Strategic Planning Process, Law Building and Approves Physical Therap',
 'date': 'October 16, 2026',
 'author': None,
 'text': 'In its fall meeting, Georgetown’s board of directors reviewed efforts underway for a new strategic p'}